# Sprint 2 — MVP Analítico
## Análise Espacial da Criminalidade no Brasil — Pré-processamento, Engenharia de Atributos e Modelo Baseline de Risco

**Aluno(a):** Cintia Nunes  
**Curso:** Ciência da Computação — UFFS  
**Disciplina:** Planejamento e Gestão de Projetos — Prof. Fabrício Herpich

Este notebook dá continuidade à Sprint 1 (EDA). Ele:

1. carrega os dados processados exportados na Sprint 1 (`sinesp_resumo_2015_2026.csv` e, opcionalmente, `cobertura_2015_2026.csv`);
2. constrói um **painel UF × Ano**, com um indicador escolhido como alvo e os demais indicadores como atributos auxiliares;
3. consulta a população por UF e ano no SIDRA/IBGE e calcula taxas por 100 mil habitantes;
4. faz engenharia de atributos temporais (defasagem, variação percentual);
5. define uma variável de **risco** (Baixo / Médio / Alto), a partir de tercis da taxa do indicador-alvo;
6. separa treino e teste **por ano** (para simular uso prospectivo do modelo, evitando vazamento temporal);
7. treina e compara modelos baseline de classificação (Dummy, Regressão Logística e Random Forest);
8. avalia com acurácia, F1-macro e matriz de confusão;
9. exporta o modelo treinado (`joblib`) e a tabela de métricas.

> **Versão ajustada (pós-Sprint 2):** esta versão inclui a UF como atributo categórico (one-hot), regularização no Random Forest e na Regressão Logística, uma seção de **validação walk-forward** (seção 11.1) para checar a robustez do corte treino/teste, e uma **versão binária do risco** (Baixo vs. Não-Baixo, seção 13.1) para reduzir a instabilidade causada pela classe "Alto" ter poucos casos.

Seguindo a DSRM, esta sprint corresponde à fase de **Design e Desenvolvimento**: o ciclo de vida de Machine Learning (preparo dos dados → definição do alvo → treino → avaliação → exportação do artefato) é conduzido de forma iterativa e documentado passo a passo.

## ✅ Versão corrigida (v3) — sem vazamento de dados

Correções em relação à versão anterior:

1. **`variacao_pct_alvo`** agora é a variação entre os dois anos *anteriores* `(lag1 − lag2) / lag2`. Antes usava a taxa do próprio ano, o que permitia reconstruir o alvo.
2. **`total_geral` e `taxa_total_100k`** agora **não incluem** o indicador-alvo.
3. **Tercis (limites de risco)** calculados só com os anos de **treino** (antes usavam toda a série, inclusive o teste).
4. **População de 2022 e 2023** (sem estimativa na tabela 6579 do SIDRA) é **interpolada** por UF, em vez de descartar os anos — assim o "ano anterior" é de fato o ano anterior.
5. O **indicador-alvo** é uma variável simples (célula da seção 3), sem widget.

> As métricas deste notebook **não são comparáveis** às da versão anterior: as anteriores estavam infladas pelo vazamento. O modelo agora estima o risco do ano a partir do histórico do indicador-alvo e das contagens dos *outros* indicadores no mesmo ano.


## 1. Preparação do ambiente

In [ ]:
import sys
import subprocess
import importlib.util

pacotes = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "requests": "requests",
    "scikit-learn": "scikit-learn",
    "joblib": "joblib",
    "ipywidgets": "ipywidgets",
}

faltando = [
    pacote
    for modulo, pacote in pacotes.items()
    if importlib.util.find_spec(modulo.replace("-", "_")) is None
]

if faltando:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltando])

print("Ambiente preparado.")

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import joblib
import ipywidgets as widgets
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

PASTA_PROCESSADOS = Path("dados/processados")
PASTA_RESULTADOS = Path("resultados")
PASTA_MODELOS = Path("modelos")

for pasta in [PASTA_PROCESSADOS, PASTA_RESULTADOS, PASTA_MODELOS]:
    pasta.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

print("Pastas preparadas.")

## 2. Carregamento dos dados processados na Sprint 1

Este notebook parte do arquivo `sinesp_resumo_2015_2026.csv`, exportado ao final do notebook de EDA
(Sprint 1). Ele já contém os dados resumidos por **ano, mês, UF, indicador e abrangência**, com a
coluna `valor` (que prioriza `total_vitima` e usa `total` como alternativa).

Se disponível, o arquivo `cobertura_2015_2026.csv` também é carregado, para identificar anos com
cobertura parcial (menos de 12 meses), que serão excluídos da modelagem.

> Faça o upload dos arquivos exportados na Sprint 1 quando solicitado.

In [ ]:
USAR_GOOGLE_DRIVE = False
PASTA_GOOGLE_DRIVE = "/content/drive/MyDrive/Sinesp/processados"

if USAR_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    caminho_resumo = Path(PASTA_GOOGLE_DRIVE) / "sinesp_resumo_2015_2026.csv"
    caminho_cobertura = Path(PASTA_GOOGLE_DRIVE) / "cobertura_2015_2026.csv"
else:
    from google.colab import files

    print("Selecione 'sinesp_resumo_2015_2026.csv' (obrigatório) e, se tiver, 'cobertura_2015_2026.csv'.")
    arquivos_enviados = files.upload()

    caminho_resumo = None
    caminho_cobertura = None

    for nome in arquivos_enviados.keys():
        if "resumo" in nome.lower():
            caminho_resumo = Path(nome)
        elif "cobertura" in nome.lower():
            caminho_cobertura = Path(nome)

if caminho_resumo is None or not caminho_resumo.exists():
    raise RuntimeError("O arquivo de resumo da Sprint 1 (sinesp_resumo_2015_2026.csv) é obrigatório.")

resumo_sinesp = pd.read_csv(caminho_resumo, encoding="utf-8-sig")

if caminho_cobertura is not None and caminho_cobertura.exists():
    cobertura = pd.read_csv(caminho_cobertura, encoding="utf-8-sig")
    anos_parciais = cobertura.loc[cobertura["mes_final"] < 12, "ano"].tolist()
else:
    cobertura = None
    anos_parciais = []

print("Registros carregados:", len(resumo_sinesp))
print("Anos disponíveis:", sorted(resumo_sinesp["ano"].unique()))
print("Anos com cobertura parcial (serão excluídos da modelagem):", anos_parciais)

resumo_sinesp = resumo_sinesp[~resumo_sinesp["ano"].isin(anos_parciais)].copy()

print("Registros após remover anos parciais:", len(resumo_sinesp))

## 3. Escolha do indicador-alvo

O indicador-alvo é aquele cuja taxa por 100 mil habitantes será usada para definir o **nível de
risco** (variável a ser prevista). Os demais indicadores entram como atributos (features) auxiliares
do modelo, junto com a defasagem (lag) do próprio indicador-alvo.

In [ ]:
# Indicador-alvo como variável simples (sem widget), para não "voltar" ao 1º item ao executar tudo.
# Para gerar OUTRO modelo: troque o valor abaixo e execute de novo A PARTIR DESTA CÉLULA.
INDICADOR_ALVO = "Arma de Fogo Apreendida"

indicadores_disponiveis = sorted(resumo_sinesp["indicador"].dropna().unique().tolist())
print("Indicadores disponíveis:")
for ind in indicadores_disponiveis:
    print(" -", ind)

In [ ]:
if INDICADOR_ALVO not in indicadores_disponiveis:
    raise RuntimeError(f"Indicador-alvo inválido: {INDICADOR_ALVO!r}. Use um dos listados acima.")
print("Indicador-alvo selecionado:", INDICADOR_ALVO)

## 4. Construção do painel UF × Ano

O painel agrega o `valor` por UF, ano e indicador (somando as abrangências disponíveis) e depois
"espalha" os indicadores em colunas, de forma que cada linha represente uma combinação
**(UF, Ano)** e cada coluna seja o total de um indicador naquele ano. Essa é a base da engenharia
de atributos: um mesmo ano/UF passa a ter, como atributos, os totais de todos os tipos de
ocorrência registrados.

In [ ]:
base_uf_ano = (
    resumo_sinesp
    .groupby(["ano", "uf", "indicador"], as_index=False)["valor"]
    .sum()
)

painel = base_uf_ano.pivot_table(
    index=["ano", "uf"],
    columns="indicador",
    values="valor",
    fill_value=0,
).reset_index()

painel.columns.name = None

print("Dimensão do painel UF x Ano:", painel.shape)
display(painel.head())

## 5. População por UF e ano (SIDRA/IBGE)

A população é consultada para todos os anos presentes no painel, permitindo calcular taxas por
100 mil habitantes de forma consistente ao longo da série histórica (e não apenas para um único
ano, como na Sprint 1).

In [ ]:
SIDRA_TABELA_POP = 6579
SIDRA_VARIAVEL_POP = 9324

CODIGO_UF = {
    11: "RO", 12: "AC", 13: "AM", 14: "RR", 15: "PA", 16: "AP", 17: "TO",
    21: "MA", 22: "PI", 23: "CE", 24: "RN", 25: "PB", 26: "PE", 27: "AL",
    28: "SE", 29: "BA", 31: "MG", 32: "ES", 33: "RJ", 35: "SP", 41: "PR",
    42: "SC", 43: "RS", 50: "MS", 51: "MT", 52: "GO", 53: "DF",
}

def carregar_populacao_uf(ano):
    url = (
        "https://apisidra.ibge.gov.br/values/"
        f"t/{SIDRA_TABELA_POP}/n3/all/"
        f"v/{SIDRA_VARIAVEL_POP}/p/{ano}"
    )
    try:
        resposta = requests.get(url, timeout=60)
        resposta.raise_for_status()
        dados = resposta.json()
    except Exception as erro:
        print(f"Não foi possível consultar a população de {ano}: {erro}")
        return pd.DataFrame(columns=["uf", "ano", "populacao"])

    linhas = []
    for registro in dados[1:]:
        try:
            codigo = int(registro["D1C"])
            populacao = float(registro["V"])
        except (KeyError, ValueError):
            continue
        uf = CODIGO_UF.get(codigo)
        if uf is not None:
            linhas.append({"uf": uf, "ano": int(ano), "populacao": populacao})

    return pd.DataFrame(linhas)

anos_painel = sorted(painel["ano"].unique().tolist())

populacoes = []
for ano in anos_painel:
    print(f"Consultando população de {ano}...")
    populacoes.append(carregar_populacao_uf(ano))

pop_uf_ano = pd.concat(populacoes, ignore_index=True) if populacoes else pd.DataFrame(columns=["uf", "ano", "populacao"])

print("\nAnos com população disponível:", sorted(pop_uf_ano["ano"].unique().tolist()) if not pop_uf_ano.empty else "nenhum")
display(pop_uf_ano.head())# CORREÇÃO: a tabela 6579 não traz alguns anos (ex.: 2022, ano do Censo, e 2023). Em vez de perder
# esses anos (o que quebrava a sequência do "ano anterior"), a população é interpolada
# linearmente por UF entre os anos vizinhos disponíveis. As linhas interpoladas ficam marcadas.
pop_uf_ano["ano"] = pd.to_numeric(pop_uf_ano["ano"]).astype(int)
pop_uf_ano["populacao"] = pd.to_numeric(pop_uf_ano["populacao"], errors="coerce")
ufs_painel = sorted(painel["uf"].unique().tolist())
grade = pd.MultiIndex.from_product([ufs_painel, anos_painel], names=["uf", "ano"]).to_frame(index=False)
pop_uf_ano = grade.merge(pop_uf_ano, on=["uf", "ano"], how="left").sort_values(["uf", "ano"])
pop_uf_ano["populacao_interpolada"] = pop_uf_ano["populacao"].isna()
pop_uf_ano["populacao"] = pop_uf_ano.groupby("uf")["populacao"].transform(
    lambda s: s.interpolate(limit_area="inside")
)
print("Anos com população interpolada:", sorted(pop_uf_ano.loc[pop_uf_ano["populacao_interpolada"], "ano"].unique().tolist()))

## 6. Cálculo das taxas por 100 mil habitantes

A taxa é calculada para o indicador-alvo (usada como base do risco) e também para o total geral
de ocorrências do ano/UF (usado como atributo de contexto).

In [ ]:
painel = painel.merge(pop_uf_ano[["uf", "ano", "populacao", "populacao_interpolada"]], on=["uf", "ano"], how="left")

colunas_indicadores = [
    c for c in painel.columns if c not in ["ano", "uf", "populacao", "populacao_interpolada"]
]

if INDICADOR_ALVO not in painel.columns:
    raise RuntimeError(f"O indicador-alvo '{INDICADOR_ALVO}' não foi encontrado no painel.")

# CORREÇÃO (vazamento): o total geral NÃO inclui o indicador-alvo. Antes, ele continha o alvo
# e permitia ao modelo "enxergar" parte da resposta.
outros_indicadores = [c for c in colunas_indicadores if c != INDICADOR_ALVO]
painel["total_geral"] = painel[outros_indicadores].sum(axis=1)

painel["taxa_alvo_100k"] = painel[INDICADOR_ALVO] / painel["populacao"] * 100_000
painel["taxa_total_100k"] = painel["total_geral"] / painel["populacao"] * 100_000

print("Registros sem população (taxa não calculável):", painel["populacao"].isna().sum())
painel = painel.dropna(subset=["populacao"]).copy()

display(
    painel[["ano", "uf", INDICADOR_ALVO, "total_geral", "populacao", "taxa_alvo_100k", "taxa_total_100k"]]
    .sort_values(["ano", "uf"])
    .head(10)
)

## 7. Engenharia de atributos temporais

Para cada UF, calcula-se a taxa do indicador-alvo no ano anterior (`lag`) e a variação percentual
em relação a esse ano anterior. Esses atributos ajudam o modelo a capturar tendência, e não apenas
o nível absoluto de um único ano.

In [ ]:
painel = painel.sort_values(["uf", "ano"]).reset_index(drop=True)

painel["taxa_alvo_100k_lag1"] = painel.groupby("uf")["taxa_alvo_100k"].shift(1)
painel["taxa_alvo_100k_lag2"] = painel.groupby("uf")["taxa_alvo_100k"].shift(2)

# CORREÇÃO (vazamento): antes, a variação era (taxa do ano - lag1) / lag1, ou seja, usava a
# própria taxa do ano que o modelo deve prever (com lag1 + variação dava para recalcular o alvo).
# Agora a variação compara os dois anos ANTERIORES: (lag1 - lag2) / lag2.
variacao = (
    (painel["taxa_alvo_100k_lag1"] - painel["taxa_alvo_100k_lag2"])
    / painel["taxa_alvo_100k_lag2"]
) * 100
painel["variacao_pct_alvo"] = variacao.replace([np.inf, -np.inf], np.nan)

display(
    painel[["uf", "ano", "taxa_alvo_100k", "taxa_alvo_100k_lag1", "taxa_alvo_100k_lag2", "variacao_pct_alvo"]]
    .sort_values(["uf", "ano"])
    .head(15)
)

## 8. Definição da variável de risco (alvo de classificação)

A taxa do indicador-alvo é convertida em três classes de risco (**Baixo, Médio, Alto**), a partir dos
tercis calculados sobre toda a série histórica disponível. Essa é a variável que os modelos vão
aprender a prever a partir dos demais atributos (outros indicadores do mesmo ano, população,
taxa geral e o histórico do próprio indicador-alvo).

In [ ]:
# Separação temporal definida ANTES dos tercis (só anos com lag1 e lag2 entram na modelagem).
anos_modelo = sorted(painel.dropna(subset=["taxa_alvo_100k_lag1", "taxa_alvo_100k_lag2"])["ano"].unique().tolist())
PROPORCAO_TESTE_ANOS = 0.25
n_anos_teste = max(1, round(len(anos_modelo) * PROPORCAO_TESTE_ANOS))
anos_teste = anos_modelo[-n_anos_teste:]
anos_treino = [a for a in anos_modelo if a not in anos_teste]

# CORREÇÃO (vazamento): os tercis são calculados SÓ com os anos de treino. Antes usavam a série
# inteira, incluindo os anos de teste. O teste é classificado com os limites aprendidos no treino.
base_tercis = painel.loc[painel["ano"].isin(anos_treino), "taxa_alvo_100k"]
tercil_1 = base_tercis.quantile(1 / 3)
tercil_2 = base_tercis.quantile(2 / 3)

def classificar_risco(taxa):
    if pd.isna(taxa):
        return np.nan
    if taxa <= tercil_1:
        return "Baixo"
    if taxa <= tercil_2:
        return "Médio"
    return "Alto"

painel["risco"] = painel["taxa_alvo_100k"].apply(classificar_risco)

print("Tercil 1 (treino):", round(tercil_1, 2))
print("Tercil 2 (treino):", round(tercil_2, 2))
print("\nDistribuição no TREINO:")
display(painel.loc[painel["ano"].isin(anos_treino), "risco"].value_counts())
print("Distribuição no TESTE (pode ser diferente: a criminalidade muda ao longo do tempo):")
display(painel.loc[painel["ano"].isin(anos_teste), "risco"].value_counts())

painel["risco"].value_counts().plot(kind="bar", figsize=(6, 4), title=f"Distribuição das classes de risco — {INDICADOR_ALVO}")
plt.ylabel("Quantidade de registros (UF x Ano)")
plt.tight_layout()
plt.show()

## 9. Separação treino/teste por ano

Para simular o uso real do modelo (estimar o risco de anos futuros com base em anos anteriores),
a separação entre treino e teste é feita **por ano**, e não aleatoriamente: os anos mais recentes
completos ficam reservados para teste. Registros sem `lag` (primeiro ano de cada UF na série) são
descartados, pois não possuem histórico anterior disponível.

In [ ]:
painel_modelo = painel.dropna(subset=["taxa_alvo_100k_lag1", "taxa_alvo_100k_lag2", "risco"]).copy()

print("Anos disponíveis para modelagem:", anos_modelo)
print("Anos de treino:", anos_treino)
print("Anos de teste:", anos_teste)

treino = painel_modelo[painel_modelo["ano"].isin(anos_treino)].copy()
teste = painel_modelo[painel_modelo["ano"].isin(anos_teste)].copy()

# A UF é mantida como atributo categórico (one-hot). O alvo, sua taxa e o lag2 ficam de fora.
colunas_excluidas = ["ano", "risco", "taxa_alvo_100k", "taxa_alvo_100k_lag2", "populacao_interpolada", INDICADOR_ALVO]
atributos = [c for c in painel_modelo.columns if c not in colunas_excluidas]

ATRIBUTOS_CATEGORICOS = ["uf"]
ATRIBUTOS_NUMERICOS = [c for c in atributos if c not in ATRIBUTOS_CATEGORICOS]

X_treino, y_treino = treino[atributos], treino["risco"]
X_teste, y_teste = teste[atributos], teste["risco"]

print("\nRegistros de treino:", len(X_treino))
print("Registros de teste:", len(X_teste))
print("Atributos utilizados:", len(atributos))

## 10. Pipeline de pré-processamento

Todos os atributos são numéricos (contagens de ocorrências, população e taxas). O pré-processamento
trata valores ausentes (imputação pela mediana) e padroniza a escala das variáveis, o que é
particularmente importante para a Regressão Logística.

In [ ]:
from sklearn.preprocessing import FunctionTransformer

def limpar_infinitos(X):
    X = np.asarray(X, dtype=float)
    return np.where(np.isinf(X), np.nan, X)

pre_processamento = ColumnTransformer(
    transformers=[
        ("numericas", Pipeline(steps=[
            ("remover_infinitos", FunctionTransformer(limpar_infinitos, feature_names_out="one-to-one")),
            ("imputador", SimpleImputer(strategy="median")),
            ("padronizador", StandardScaler()),
        ]), ATRIBUTOS_NUMERICOS),
        ("categoricas", Pipeline(steps=[
            ("imputador_uf", SimpleImputer(strategy="most_frequent")),
            ("onehot_uf", OneHotEncoder(handle_unknown="ignore")),
        ]), ATRIBUTOS_CATEGORICOS),
    ]
)


## 11. Modelos baseline e comparação

São comparados três modelos:

- **Dummy (estratificado)**: referência mínima, prevê respeitando apenas a proporção das classes.
- **Regressão Logística**: modelo linear simples e interpretável.
- **Random Forest**: modelo não linear, capaz de capturar interações entre indicadores.

A comparação usa acurácia e **F1-macro** (mais adequado que acurácia simples quando as classes
podem ficar desbalanceadas entre treino e teste).

In [ ]:
# Regularização mais forte na Regressão Logística (C menor) e limites de
# complexidade no Random Forest (max_depth, min_samples_leaf), para reduzir
# overfitting dado o número pequeno de linhas de treino disponível.
modelos = {
    "Dummy (estratificado)": DummyClassifier(strategy="stratified", random_state=RANDOM_STATE),
    "Regressão Logística": LogisticRegression(
        max_iter=2000, C=0.5, random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=4,
        min_samples_leaf=3,
        max_features="sqrt",
        random_state=RANDOM_STATE,
    ),
}

resultados = []
pipelines_treinados = {}

for nome, modelo in modelos.items():
    pipeline = Pipeline(steps=[
        ("pre_processamento", pre_processamento),
        ("modelo", modelo),
    ])

    pipeline.fit(X_treino, y_treino)
    y_previsto = pipeline.predict(X_teste)

    acuracia = accuracy_score(y_teste, y_previsto)
    f1_macro = f1_score(y_teste, y_previsto, average="macro")

    resultados.append({
        "modelo": nome,
        "acuracia": round(acuracia, 4),
        "f1_macro": round(f1_macro, 4),
    })

    pipelines_treinados[nome] = pipeline

    print(f"\n=== {nome} ===")
    print(classification_report(y_teste, y_previsto, zero_division=0))

tabela_resultados = pd.DataFrame(resultados).sort_values("f1_macro", ascending=False)
display(tabela_resultados)

## 11.1 Validação walk-forward (robustez extra)

A comparação anterior usa um único corte treino/teste (últimos anos como teste). Com poucos anos
disponíveis, esse único corte pode não ser representativo — o resultado pode mudar bastante
dependendo de qual ano específico caiu no teste.

Para checar a robustez dos resultados, esta seção repete o treino/teste em uma janela deslizante
("walk-forward"): a cada rodada, treina com os anos `[..., k]` e testa no ano `k+1` (o ano
imediatamente seguinte), avançando a janela até o fim da série. As métricas de cada rodada são
depois agregadas (média e desvio-padrão), dando uma estimativa mais confiável do que um único
corte de teste.

In [ ]:
def avaliar_walk_forward(modelos_dict, painel_modelo, atributos, pre_processamento_template,
                          ano_minimo_treino=2):
    """
    Executa validação walk-forward: a cada rodada, treina com todos os anos
    anteriores a `ano_teste` e testa em `ano_teste`. `ano_minimo_treino` é o
    número mínimo de anos distintos exigido no treino antes de começar a avaliar.
    """
    anos = sorted(painel_modelo["ano"].unique().tolist())
    linhas_resultado = []

    for i, ano_teste in enumerate(anos):
        anos_treino_rodada = anos[:i]
        if len(anos_treino_rodada) < ano_minimo_treino:
            continue

        treino_r = painel_modelo[painel_modelo["ano"].isin(anos_treino_rodada)]
        teste_r = painel_modelo[painel_modelo["ano"] == ano_teste]

        if teste_r.empty or treino_r.empty:
            continue
        # Pula rodadas em que o treino não tem pelo menos 2 classes presentes.
        if treino_r["risco"].nunique() < 2:
            continue

        X_tr, y_tr = treino_r[atributos], treino_r["risco"]
        X_te, y_te = teste_r[atributos], teste_r["risco"]

        for nome_modelo, modelo in modelos_dict.items():
            # Cada rodada precisa de uma cópia "fresca" do pré-processamento,
            # senão o StandardScaler/OneHotEncoder ficaria "vazando" estatísticas
            # entre rodadas.
            pipeline_r = Pipeline(steps=[
                ("pre_processamento", ColumnTransformer(
                    transformers=[t for t in pre_processamento_template.transformers]
                )),
                ("modelo", modelo),
            ])
            try:
                pipeline_r.fit(X_tr, y_tr)
                y_pred_r = pipeline_r.predict(X_te)
            except Exception as erro:
                print(f"Rodada {ano_teste} / {nome_modelo} falhou: {erro}")
                continue

            linhas_resultado.append({
                "ano_teste": ano_teste,
                "n_treino": len(X_tr),
                "n_teste": len(X_te),
                "modelo": nome_modelo,
                "acuracia": accuracy_score(y_te, y_pred_r),
                "f1_macro": f1_score(y_te, y_pred_r, average="macro", zero_division=0),
            })

    return pd.DataFrame(linhas_resultado)


# Reconstrói modelos "zerados" (sem fit anterior) para reuso na validação walk-forward
modelos_wf = {
    "Dummy (estratificado)": DummyClassifier(strategy="stratified", random_state=RANDOM_STATE),
    "Regressão Logística": LogisticRegression(max_iter=2000, C=0.5, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=4, min_samples_leaf=3,
        max_features="sqrt", random_state=RANDOM_STATE,
    ),
}

resultados_wf = avaliar_walk_forward(modelos_wf, painel_modelo, atributos, pre_processamento)

display(resultados_wf)

if not resultados_wf.empty:
    resumo_wf = (
        resultados_wf
        .groupby("modelo")[["acuracia", "f1_macro"]]
        .agg(["mean", "std"])
        .round(4)
    )
    print("\nResumo da validação walk-forward (média ± desvio-padrão entre rodadas):")
    display(resumo_wf)
else:
    print("Não foi possível rodar validação walk-forward (poucos anos disponíveis).")


## 12. Matriz de confusão do melhor modelo

In [ ]:
melhor_modelo_nome = tabela_resultados.iloc[0]["modelo"]
melhor_pipeline = pipelines_treinados[melhor_modelo_nome]

print("Melhor modelo (maior F1-macro):", melhor_modelo_nome)

y_previsto_melhor = melhor_pipeline.predict(X_teste)

matriz = confusion_matrix(y_teste, y_previsto_melhor, labels=["Baixo", "Médio", "Alto"])

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(matriz, display_labels=["Baixo", "Médio", "Alto"]).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Matriz de confusão — {melhor_modelo_nome}\n{INDICADOR_ALVO}")
plt.tight_layout()
plt.show()

## 13. Importância dos atributos (Random Forest)

Quando o Random Forest está entre os modelos treinados, é possível observar quais atributos mais
contribuem para a previsão do risco — por exemplo, se outros tipos de ocorrência, a população ou o
histórico do próprio indicador-alvo (`lag`) pesam mais na classificação.

In [ ]:
if "Random Forest" in pipelines_treinados:
    rf_pipeline = pipelines_treinados["Random Forest"]
    rf_modelo = rf_pipeline.named_steps["modelo"]

    # Como a UF agora passa por one-hot, o número de colunas após o
    # pré-processamento é diferente do número de atributos originais.
    # Usamos get_feature_names_out() para recuperar os nomes corretos.
    nomes_transformados = rf_pipeline.named_steps["pre_processamento"].get_feature_names_out()

    importancias = pd.DataFrame({
        "atributo": nomes_transformados,
        "importancia": rf_modelo.feature_importances_,
    }).sort_values("importancia", ascending=False).head(15)

    display(importancias)

    importancias.sort_values("importancia").plot(
        x="atributo", y="importancia", kind="barh", figsize=(9, 7), legend=False,
        title=f"Atributos mais relevantes — Random Forest ({INDICADOR_ALVO})",
    )
    plt.xlabel("Importância")
    plt.tight_layout()
    plt.show()
else:
    print("Random Forest não está entre os modelos treinados nesta execução.")


## 14. Exportação do modelo e das métricas

O melhor pipeline (pré-processamento + modelo) é exportado com `joblib`, junto com metadados sobre
o experimento (indicador-alvo, atributos utilizados, anos de treino/teste) e a tabela de métricas
comparativas. Esses arquivos devem ser versionados no GitHub, conforme a entrega da Sprint 2.

In [ ]:
import sklearn
nome_arquivo_indicador = (
    INDICADOR_ALVO.lower()
    .replace(" ", "_")
    .replace("/", "_")
)

arquivo_modelo = PASTA_MODELOS / f"modelo_risco_{nome_arquivo_indicador}.joblib"
arquivo_metricas = PASTA_RESULTADOS / f"metricas_modelos_{nome_arquivo_indicador}.csv"

metadados = {
    "pipeline": melhor_pipeline,
    "modelo": melhor_modelo_nome,
    "indicador_alvo": INDICADOR_ALVO,
    "atributos": atributos,
    "anos_treino": anos_treino,
    "anos_teste": anos_teste,
    "tercil_1": tercil_1,
    "tercil_2": tercil_2,
    "classes": ["Baixo", "Médio", "Alto"],
    "sklearn_version": sklearn.__version__,
    "versao_pipeline": "v3-sem-vazamento",
}

joblib.dump(metadados, arquivo_modelo)
tabela_resultados.to_csv(arquivo_metricas, index=False, encoding="utf-8-sig")

print("Modelo exportado em:", arquivo_modelo)
print("Métricas exportadas em:", arquivo_metricas)

## 13.1 Versão alternativa: risco binário (Baixo vs. Não-Baixo)

Com apenas 5 casos de "Alto" no teste, essa classe concentra boa parte da instabilidade das
métricas (pequenas mudanças de previsão causam grandes saltos em precisão/recall). Como
verificação extra de robustez, esta seção reagrupa "Médio" e "Alto" em uma única classe
("Não-Baixo") e recalcula os modelos como um problema binário. Isso não substitui a análise
de 3 classes — é um cenário complementar para reportar no artigo, caso a instabilidade da
classe "Alto" seja um ponto de atenção.

In [ ]:
def binarizar_risco(valor):
    return "Baixo" if valor == "Baixo" else "Não-Baixo"

y_treino_bin = y_treino.apply(binarizar_risco)
y_teste_bin = y_teste.apply(binarizar_risco)

resultados_bin = []
pipelines_bin = {}

modelos_bin = {
    "Dummy (estratificado)": DummyClassifier(strategy="stratified", random_state=RANDOM_STATE),
    "Regressão Logística": LogisticRegression(max_iter=2000, C=0.5, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=4, min_samples_leaf=3,
        max_features="sqrt", random_state=RANDOM_STATE,
    ),
}

for nome, modelo in modelos_bin.items():
    pipeline_b = Pipeline(steps=[
        ("pre_processamento", ColumnTransformer(
            transformers=[t for t in pre_processamento.transformers]
        )),
        ("modelo", modelo),
    ])
    pipeline_b.fit(X_treino, y_treino_bin)
    y_previsto_b = pipeline_b.predict(X_teste)

    resultados_bin.append({
        "modelo": nome,
        "acuracia": round(accuracy_score(y_teste_bin, y_previsto_b), 4),
        "f1_macro": round(f1_score(y_teste_bin, y_previsto_b, average="macro"), 4),
    })
    pipelines_bin[nome] = pipeline_b

    print(f"\n=== {nome} (binário: Baixo vs Não-Baixo) ===")
    print(classification_report(y_teste_bin, y_previsto_b, zero_division=0))

tabela_resultados_bin = pd.DataFrame(resultados_bin).sort_values("f1_macro", ascending=False)
display(tabela_resultados_bin)


## 15. Conclusões da Sprint 2

Após executar o notebook, utilize os resultados para registrar no artigo científico:

1. qual indicador foi utilizado como alvo e por quê;
2. quais atributos compõem o painel (outros indicadores, população, taxa geral, lag e variação percentual);
3. como foi feita a separação temporal entre treino e teste, e por que essa escolha evita vazamento de dados;
4. como os modelos baseline (Dummy, Regressão Logística, Random Forest) se compararam em acurácia e F1-macro;
5. quais atributos mais influenciaram o modelo de melhor desempenho;
6. quais limitações foram identificadas (poucos anos disponíveis, classes de risco definidas por tercis, dependência da qualidade do Sinesp VDE).

## 16. Limitações

- O número de anos disponíveis é pequeno para um problema de série temporal, o que limita a robustez da divisão treino/teste.
- As classes de risco (tercis) são uma simplificação: não correspondem a um critério oficial de classificação de risco em segurança pública.
- A ausência de população para algum ano/UF remove essas observações da modelagem.
- O modelo é um **baseline**: etapas futuras podem incluir validação cruzada temporal (walk-forward), ajuste de hiperparâmetros e testes de autocorrelação espacial (Moran/LISA) como atributos adicionais.
- Este notebook não estabelece relação causal para a criminalidade; trata-se de um modelo preditivo/exploratório.